In [1]:
from pyspark.sql import SparkSession
import os as os

OIDC_TOKEN = os.environ["OIDC_TOKEN"]
spark = (SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002")
.config("spark.sql.catalog.test_az.scope", "openid")
.config("spark.sql.catalog.test_az.header.X-Iceberg-Access-Delegation", "vended-credentials")
.config("spark.sql.catalog.test_az", "org.apache.iceberg.spark.SparkCatalog")
.config("spark.sql.catalog.test_az.type", "rest")
.config("spark.sql.catalog.test_az.uri", "https://westmidspolarisazure.click/api/catalog")
.config("spark.sql.catalog.test_az.cache-enabled", "false")
.config("spark.sql.catalog.test_az.warehouse", "azure")
.config("spark.sql.catalog.test_az.token", OIDC_TOKEN)
.config("spark.sql.catalog.test_az.oauth2-server-uri", "https://emrysoauth.click/realms/polaris/protocol/openid-connect/token")
 .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
 .config("spark.driver.allowMultipleContexts", "true")
 .config("spark.sql.iceberg.planning.preserve-data-grouping", "true")
 .config("spark.sql.sources.v2.bucketing.enabled", "true")
 .config("spark.sql.sources.bucketing.enabled", "true")
).getOrCreate()

print("Finished initializing spark session")


Finished initializing spark session


/opt/conda/lib/python3.11/site-packages/pyspark/sql/connect/conf.py:64: UserWarning: Failed to set spark.sql.extensions to Some(org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions) due to [CANNOT_MODIFY_CONFIG] Cannot modify the value of the Spark config: "spark.sql.extensions".
See also 'https://spark.apache.org/docs/latest/sql-migration-guide.html#ddl-statements'. SQLSTATE: 46110
  warnings.warn(warn)


In [2]:
df_test = spark.createDataFrame([["test", 1]], ['name', 'num'])
print("Initialized test data")

Initialized test data


In [3]:
from datetime import datetime

In [18]:
discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
az_namespace = "test_az.namespaceph"
az_fqn = f"{az_namespace}.smoketesttable_" + discriminator
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {az_namespace}")
print(f"Created namespace {az_namespace}")
print("Creating table")
spark.sql(f"CREATE TABLE IF NOT EXISTS {az_fqn} (name String, num Integer) USING ICEBERG")
print(f"Writing to ${az_fqn} ...")
df_test.writeTo(az_fqn).append()
print(f"Wrote {az_fqn} to Azure")

Created namespace test_az.namespaceph
Creating table
Writing to $test_az.namespaceph.smoketesttable_28August2026_10_01 ...
Wrote test_az.namespaceph.smoketesttable_28August2026_10_01 to Azure


In [12]:
spark.read.table("test_az.namespaceph.test_table").show()

+------+
| value|
+------+
| apple|
| peach|
|banana|
|cherry|
+------+



In [11]:
df = spark.createDataFrame([["peach"]], ['value'])
df.writeTo("test_az.namespaceph.test_table").append()
print("done")